In [21]:
### 1. Load and Prepare Image
from pathlib import Path
import numpy as np
import tifffile as tiff
from tqdm.notebook import tqdm

# Set your input path here
input_path = Path(r"D:\Thesis_images\Final_dataset\all\test\PN_KC_488_568_Brain2_CalyxR_LSMPlus_16bit.ome.tif")

def load_and_standardize(path):
    """Loads a 4D image and standardizes its dimension order to (Z, C, Y, X)."""
    img = tiff.imread(str(path))
    if img.ndim != 4:
        raise ValueError(f"Expected a 4D image, got shape {img.shape}")

    # Standardize to (Z, C, Y, X)
    if img.shape[0] == 2:          # (C, Z, Y, X) -> (Z, C, Y, X)
        img = np.transpose(img, (1, 0, 2, 3))
    elif img.shape[1] == 2:        # (Z, C, Y, X) - already correct
        pass
    else:
        raise ValueError(f"Cannot infer channel axis from shape {img.shape}")

    return img

# Load the full stack
img_stack = load_and_standardize(input_path)
print("Full stack shape:", img_stack.shape)

Full stack shape: (76, 2, 1101, 1101)


In [22]:
### 2. Apply Rolling Ball Background Subtraction
from skimage.restoration import rolling_ball

print("Applying rolling ball background subtraction to the full stack...")
img_bg_sub = np.zeros_like(img_stack, dtype=np.float32)
ball_radius = 50 # You can adjust this value

# Process each slice in the Z-stack
for z in tqdm(range(img_stack.shape[0]), desc="Z-slices"):
    # Channel 0: Red, Channel 1: Green
    for c, channel_name in enumerate(['Red', 'Green']):
        background = rolling_ball(img_stack[z, c, :, :], radius=ball_radius)
        img_bg_sub[z, c, :, :] = img_stack[z, c, :, :] - background

print("Background subtraction complete for the entire stack.")

Applying rolling ball background subtraction to the full stack...


Z-slices:   0%|          | 0/76 [00:00<?, ?it/s]

Background subtraction complete for the entire stack.


In [23]:
### 3. Sharpening with Difference of Gaussians
from skimage.filters import gaussian
import numpy as np

def sharpen_with_dog(image_stack, sigma_low=1.5, sigma_high=3.0, strength=1.0):
    """
    Applies DoG sharpening to a full 3D stack (Z, C, Y, X).
    """
    print("\nApplying Difference of Gaussians Sharpening...")
    sharpened_stack = np.zeros_like(image_stack, dtype=np.float32)
    
    for z in tqdm(range(image_stack.shape[0]), desc="Sharpening Z-slices"):
        for c in range(image_stack.shape[1]):
            ch_slice = image_stack[z, c].astype(np.float32)
            
            # Apply Gaussian filters
            gauss_low = gaussian(ch_slice, sigma=sigma_low)
            gauss_high = gaussian(ch_slice, sigma=sigma_high)
            
            # Calculate Difference of Gaussians and clip negative values
            dog = gauss_low - gauss_high
            dog_clipped = np.clip(dog, 0, None)
            
            # Add the clipped DoG back to the original slice
            sharpened_slice = ch_slice + (dog_clipped * strength)
            sharpened_stack[z, c] = sharpened_slice
            
    print("\nSharpening complete.")
    return sharpened_stack

# --- Run the sharpening on the background-subtracted stack ---
img_final = sharpen_with_dog(img_bg_sub, strength=1.5)


Applying Difference of Gaussians Sharpening...


Sharpening Z-slices:   0%|          | 0/76 [00:00<?, ?it/s]


Sharpening complete.


In [24]:
### 4. Save the Sharpened Image
import tifffile as tiff
from pathlib import Path
import numpy as np

# --- Normalization function for saving ---
def normalize_to_float(data):
    """Scales data to the [0.0, 1.0] range for saving as float."""
    min_val = data.min()
    max_val = data.max()
    if max_val > min_val:
        return ((data - min_val) / (max_val - min_val)).astype(np.float32)
    return data.astype(np.float32)

# --- Prepare Filename ---
# Creates a name like "original_filename_dog.tif"
output_filename = input_path.stem.replace('.ome', '') + "_dog.tif"
output_path = input_path.parent / output_filename

# --- Normalize and Save ---
print(f"Normalizing final image to range [0.0, 1.0]...")
img_to_save = normalize_to_float(img_final)

print(f"Saving sharpened image to: {output_path}")
# We save the stack in (Z, C, Y, X) format.
# imagej=True helps with metadata compatibility.
tiff.imwrite(output_path, img_to_save, imagej=True)

print("\nSave complete.")
print(f"Final saved image shape: {img_to_save.shape}")
print(f"Final saved image range: {img_to_save.min():.4f} - {img_to_save.max():.4f}")

Normalizing final image to range [0.0, 1.0]...
Saving sharpened image to: D:\Thesis_images\Final_dataset\all\test\PN_KC_488_568_Brain2_CalyxR_LSMPlus_16bit_dog.tif

Save complete.
Final saved image shape: (76, 2, 1101, 1101)
Final saved image range: 0.0000 - 1.0000
